# Lab 03.2 — Spark su Hadoop YARN

## Introduzione

**YARN** (Yet Another Resource Negotiator) è il gestore di risorse di Hadoop: lo stesso cluster può eseguire Spark, MapReduce, Hive, ...

Con `master("yarn")`:

1. il driver (questo notebook) chiede al **ResourceManager** di avviare un'applicazione;
2. su un **NodeManager** parte l'**ApplicationMaster** di Spark, che negozia i container per gli executor;
3. gli executor girano in **container YARN** con memoria e CPU assegnate.

> Questo notebook richiede la modalità YARN: `COMPOSE_PROFILES=yarn SPARK_MASTER=yarn docker compose up -d`
> ed i file creati dal notebook `01_hdfs_e_spark` (eseguilo prima, anche in modalità YARN).

UI: ResourceManager http://localhost:8088 · NodeManager http://localhost:8042

### Obiettivi

1. Avviare una `SparkSession` con master `yarn` e riconoscere l'**Application ID** YARN.
2. Capire quale **configurazione** (Hadoop e Spark) usa il driver.
3. Vedere **dove girano i task**: container del NodeManager.
4. Interrogare la **REST API del ResourceManager** per stato e risorse dell'applicazione.
5. Eseguire un job che legge da **HDFS** su YARN.
6. **Chiudere** l'applicazione e verificarne lo stato finale.

## Step 1: Avvio della SparkSession su YARN

Prima di tutto verifichiamo con `assert` che il laboratorio sia stato avviato in modalità YARN (`SPARK_MASTER=yarn`). Poi `crea_sessione` chiede al ResourceManager di avviare un'applicazione: parte l'ApplicationMaster e vengono richiesti i container per gli executor (il primo avvio richiede qualche secondo). L'**Application ID** ha il formato YARN `application_<timestamp>_<n>`.

In [ ]:
import os, socket, requests
from sessione import crea_sessione  # helper del laboratorio: con SPARK_MASTER=yarn aggiunge la configurazione YARN

# Il notebook ha senso solo in modalità YARN: altrimenti ci fermiamo con un messaggio chiaro
assert os.environ.get("SPARK_MASTER") == "yarn", "Avvia il laboratorio con: COMPOSE_PROFILES=yarn SPARK_MASTER=yarn docker compose up -d"

spark = crea_sessione("Lab03-YARN")
sc = spark.sparkContext
print("Master        :", sc.master)
print("Application ID:", sc.applicationId, " (formato YARN: application_<timestamp>_<n>)")

## Step 2: Configurazione usata

Il driver trova il ResourceManager leggendo `yarn-site.xml` dalla cartella indicata da `HADOOP_CONF_DIR`. Stampiamo alcune proprietà significative in due gruppi:

| Origine | Proprietà | Significato |
|---|---|---|
| Hadoop | `yarn.resourcemanager.hostname`, `yarn.nodemanager.resource.memory-mb`, `fs.defaultFS` | dove si trova il ResourceManager, quanta memoria offre il NodeManager, filesystem di default |
| Spark | `spark.executor.instances`, `spark.executor.memory`, `spark.yarn.am.memory`, `spark.yarn.jars` | numero e memoria degli executor, memoria dell'ApplicationMaster, posizione dei jar di Spark |

In [ ]:
print("HADOOP_CONF_DIR:", os.environ["HADOOP_CONF_DIR"])
hconf = sc._jsc.hadoopConfiguration()
# Proprietà Hadoop lette dai file XML (yarn-site.xml, core-site.xml)
for k in ["yarn.resourcemanager.hostname", "yarn.nodemanager.resource.memory-mb", "fs.defaultFS"]:
    print(f"  {k} = {hconf.get(k)}")
# Proprietà Spark impostate da crea_sessione() per la modalità YARN
for k in ["spark.executor.instances", "spark.executor.memory", "spark.yarn.am.memory", "spark.yarn.jars"]:
    print(f"  {k} = {sc.getConf().get(k)}")

## Step 3: Dove girano i task?

Gli executor sono processi nei **container YARN** del NodeManager. Eseguiamo 8 task banali (8 partizioni) che restituiscono l'hostname e la variabile `CONTAINER_ID` impostata da YARN nell'ambiente di ogni container; `distinct()` elimina i duplicati, quindi vediamo un elemento per ogni container che ha eseguito almeno un task.

In [ ]:
# 8 task leggeri: ognuno riporta host e id del container YARN in cui è stato eseguito
info = sc.parallelize(range(8), 8).map(lambda _: (socket.gethostname(), os.environ.get("CONTAINER_ID", "?"))).distinct().collect()
for host, container in sorted(info):
    print(f"host {host:12s} container {container}")

### Punti chiave

* Con YARN il **ResourceManager** assegna le risorse e il **NodeManager** esegue i container (ApplicationMaster ed executor).
* Il driver conosce il cluster solo tramite i file di configurazione in `HADOOP_CONF_DIR`.
* Gli executor sono container con memoria e vcore limitati: tutti i task girano sul NodeManager, non nel notebook.

## Step 4: Il ResourceManager vede l'applicazione

Interroghiamo la **REST API** del ResourceManager (`/ws/v1/cluster`): per la nostra applicazione leggiamo nome, coda, stato, memoria e vcore allocati e numero di container in esecuzione; dalle metriche di cluster leggiamo invece le risorse totali e quelle in uso. Gli stessi dati sono visibili nella UI su http://localhost:8088.

In [ ]:
rm = "http://resourcemanager:8088/ws/v1/cluster"   # REST API del ResourceManager
# Informazioni sulla nostra applicazione: stato, coda, risorse e container
app = requests.get(f"{rm}/apps/{sc.applicationId}").json()["app"]
print({k: app[k] for k in ("name", "queue", "state", "allocatedMB", "allocatedVCores", "runningContainers")})

# Metriche dell'intero cluster: risorse allocate rispetto al totale disponibile
metriche = requests.get(f"{rm}/metrics").json()["clusterMetrics"]
print(f"Cluster: {metriche['allocatedMB']} / {metriche['totalMB']} MB allocati, "
      f"{metriche['allocatedVirtualCores']} / {metriche['totalVirtualCores']} vcore")

## Step 5: Un job che legge da HDFS su YARN

Usiamo i file caricati nel notebook precedente (`/dati/alice.txt` e `/dati/vendite`). Il percorso senza schema punta a HDFS perché `fs.defaultFS` è letto da `HADOOP_CONF_DIR`. Il Word Count (solo parole di più di 4 lettere) e la media degli importi per regione sono eseguiti dagli executor nei container YARN: se i file non esistono, rieseguire prima `01_hdfs_e_spark`.

In [ ]:
import re
from pyspark.sql import functions as F

# Word Count su HDFS, tenendo solo le parole più lunghe di 4 lettere
top = (sc.textFile("/dati/alice.txt")
         .flatMap(lambda l: re.findall(r"[a-z]+", l.lower()))
         .filter(lambda w: len(w) > 4)
         .map(lambda w: (w, 1))
         .reduceByKey(lambda a, b: a + b)
         .takeOrdered(10, key=lambda kv: -kv[1]))
print(top)

# Parquet partizionato scritto nel notebook 01: media degli importi per regione
spark.read.parquet("/dati/vendite").groupBy("regione").agg(F.round(F.avg("importo"), 2).alias("media")).show()

### Punti chiave

* Lo stesso codice Spark gira su Standalone o YARN: cambia solo il master.
* Su YARN i dati restano su **HDFS**; gli executor li leggono dai DataNode.
* Il notebook 02 dipende dai file creati dal notebook 01 (`/dati/alice.txt`, `/dati/vendite`).

## Step 6: Chiusura

Con `spark.stop()` l'applicazione termina e YARN libera i container. Dopo una breve attesa (3 secondi) chiediamo di nuovo al ResourceManager lo stato dell'applicazione: deve essere `FINISHED`, come si vede anche nella UI.

In [ ]:
app_id = sc.applicationId   # lo salviamo prima di fermare la sessione
spark.stop()                # YARN libera ApplicationMaster ed executor
import time; time.sleep(3)  # lascia al ResourceManager il tempo di aggiornare lo stato
print(app_id, "→", requests.get(f"{rm}/apps/{app_id}").json()["app"]["state"])

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Modalità YARN | `master("yarn")`, `SPARK_MASTER=yarn` | eseguire Spark sul cluster manager di Hadoop |
| Application ID | `sc.applicationId` | identificare l'applicazione in YARN (`application_<timestamp>_<n>`) |
| Configurazione | `HADOOP_CONF_DIR`, `yarn-site.xml` | far trovare al driver il ResourceManager e HDFS |
| Risorse Spark su YARN | `spark.executor.instances`, `spark.executor.memory`, `spark.yarn.am.memory` | dimensionare executor e ApplicationMaster |
| Container degli executor | variabile `CONTAINER_ID` | vedere dove girano i task |
| REST API del ResourceManager | `/ws/v1/cluster/apps/<id>`, `/metrics` | stato dell'applicazione e risorse del cluster |
| Dati su HDFS | `sc.textFile("/dati/...")`, `spark.read.parquet(...)` | leggere dai DataNode con executor YARN |
| Chiusura | `spark.stop()` | liberare i container (stato `FINISHED`) |